# [8] Simple Approach

## Imports

In [ ]:
import env

In [ ]:
from epidec.datasets import BalancedBaggedSWUnivDaconDataset, SWUnivDaconDataset

from transformers import ElectraPreTrainedModel, ElectraModel, ElectraConfig, AutoTokenizer
from torch.utils.data import DataLoader
from torch import nn, optim
import torch

from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import sys
import gc
import re

In [ ]:
from sklearn.exceptions import UndefinedMetricWarning
import warnings

warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

In [ ]:
for test in tqdm(range(1000), desc="Testing tqdm"):
    pass

In [ ]:
# project name
PROJECT_NAME = "8_simple"

### Check GPU Availability

In [ ]:
!nvidia-smi

In [ ]:
# Set CUDA Device
device_num = 0

if torch.cuda.is_available() and device_num != -1:
    torch.cuda.set_device(device_num)
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    device_num = -1  # cpu
print(f"INFO: Using device - {device}:{device_num}")

## Load Datasets

In [ ]:
DATA_ROOT = "./data"

train_dataset = BalancedBaggedSWUnivDaconDataset(DATA_ROOT, train=True, valid_ratio=0.1, balancing_ratio=1, bagging_size=5)
valid_dataset = BalancedBaggedSWUnivDaconDataset(DATA_ROOT, valid=True, valid_ratio=0.1, balancing_ratio=1)
valid_totalset = SWUnivDaconDataset(DATA_ROOT, valid=True, valid_ratio=0.1)
test_dataset = BalancedBaggedSWUnivDaconDataset(DATA_ROOT, train=False)

print(f"INFO: Dataset loaded successfully. Train - {len(train_dataset)}, Valid - {len(valid_dataset)}, Valid Total - {len(valid_totalset)}, Test - {len(test_dataset)}")

In [ ]:
train_dataset[1]

In [ ]:
valid_dataset[0]

#### Sentence segmentation

In [ ]:
import re

def split_sentence(text):
    text = text.strip()
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    pattern = r'([.!?]+)(\s+)(?=[A-Z가-힣])'

    sentences = []
    last_end = 0

    for match in re.finditer(pattern, text):
        sentence = text[last_end:match.end()-len(match.group(2))].strip()
        if sentence:
            sentences.append(sentence)
        last_end = match.end()-len(match.group(2))

    if last_end < len(text):
        last_sentence = text[last_end:].strip()
        if last_sentence:
            sentences.append(last_sentence)

    return sentences

def split_sentence(text):
    sentences = re.split(r'(?<=[.?!])\s*', text)
    return [s for s in sentences if s]

In [ ]:
# Train set
for i in range(len(train_dataset)):
    for j in range(train_dataset.bagging_size):
        train_dataset.data[j][i] = split_sentence(train_dataset.data[j][i])

# Valid set
valid_dataset.data = [split_sentence(i) for i in valid_dataset.data]

# Test set
test_dataset.data = [split_sentence(i) for i in test_dataset.data]

In [ ]:
train_dataset[1]

In [ ]:
valid_dataset[1]

In [ ]:
test_dataset[1]

## Define Model

In [ ]:
class VectorMaxPool1d(nn.Module):
    def __init__(self, criterion='norm'):
        super().__init__()
        self.criterion = criterion

    def forward(self, x):
        if self.criterion == 'norm':
            norms = torch.norm(x, dim=-1)  # (batch_size, seq_len)
            max_indices = torch.argmax(norms, dim=-1)  # (batch_size,)

        elif self.criterion == 'max_value':
            max_values = torch.max(x, dim=-1)[0]  # (batch_size, seq_len)
            max_indices = torch.argmax(max_values, dim=-1)  # (batch_size,)

        elif self.criterion == 'sum':
            sums = torch.sum(x, dim=-1)  # (batch_size, seq_len)
            max_indices = torch.argmax(sums, dim=-1)  # (batch_size,)

        elif self.criterion == 'mean':
            means = torch.mean(x, dim=-1)  # (batch_size, seq_len)
            max_indices = torch.argmax(means, dim=-1)  # (batch_size,)

        else:
            raise ValueError(f"Unknown criterion: {self.criterion}")

        # extract the selected vectors
        batch_size = x.size(0)
        selected_vectors = x[torch.arange(batch_size), max_indices]

        return selected_vectors

In [ ]:
class WeightedVectorMaxPool1d(nn.Module):
    def __init__(self, hidden_size, num_criteria=3):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_criteria = num_criteria

        self.criterion_weights = nn.Parameter(torch.ones(num_criteria))

    def forward(self, x):
        batch_size, seq_len, hidden_size = x.size()

        scores = []

        # first criterion: norm of each vector
        if self.num_criteria >= 1:
            norms = torch.norm(x, dim=-1)  # (batch_size, seq_len)
            scores.append(norms)

        # second criterion: max value in each vector
        if self.num_criteria >= 2:
            max_values = torch.max(x, dim=-1)[0]  # (batch_size, seq_len)
            scores.append(max_values)

        # third criterion: sum of values
        if self.num_criteria >= 3:
            sums = torch.sum(x, dim=-1)  # (batch_size, seq_len)
            scores.append(sums)

        # score calculation
        weighted_scores = torch.zeros_like(scores[0])
        weights = nn.functional.softmax(self.criterion_weights, dim=0)

        for i, score in enumerate(scores):
            weighted_scores += weights[i] * score

        # select the max vector based on the weighted scores
        max_indices = torch.argmax(weighted_scores, dim=-1)  # (batch_size,)
        selected_vectors = x[torch.arange(batch_size), max_indices]

        return selected_vectors

In [ ]:
class BinaryFocalWithLogitsLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = nn.functional.binary_cross_entropy_with_logits(inputs, targets, reduction='none')

        pt = torch.sigmoid(inputs)
        pt = torch.where(targets == 1, pt, 1 - pt)

        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        focal_loss = alpha_t * (1 - pt) ** self.gamma * bce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [ ]:
base_model_id = "monologg/koelectra-small-v3-discriminator"
base_model_config = ElectraConfig.from_pretrained(base_model_id)

In [ ]:
from typing import Optional

class ElectraForNaiveTextDetection(ElectraPreTrainedModel):
    def __init__(self, config: ElectraConfig = base_model_config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config
        self.electra = ElectraModel(config)
        self.classifier = nn.Sequential(
            WeightedVectorMaxPool1d(self.config.hidden_size),
            nn.Sequential(
                nn.Dropout(0.2),
                nn.Linear(self.config.hidden_size, self.config.hidden_size // 2),
                nn.LayerNorm(self.config.hidden_size // 2),
                nn.SiLU(),
                nn.Dropout(0.25),
                nn.Linear(self.config.hidden_size // 2, 1)
            )
        )

        # 3. Initialize weights and apply final processing
        self.post_init()

    def forward(
        self,
        input_ids: list[torch.LongTensor],
        attention_mask: list[torch.Tensor],
        labels: Optional[torch.Tensor] = None,
        criterion = BinaryFocalWithLogitsLoss()
    ) -> torch.Tensor:
        hidden_states = []
        for input_id, attention_mask in zip(input_ids, attention_mask):
            # forward pass by each paragraph
            hidden_state = self.electra(input_ids=input_id, attention_mask=attention_mask).last_hidden_state[:, 0, :]  # pooling by first token (CLS token)
            # [hidden_size, sentences] => [hidden_size]
            polled = self.classifier[0](hidden_state.unsqueeze(0))
            hidden_states.append(polled)  # [1, hidden_size]

        hidden_states = torch.cat(hidden_states, dim=0)  # [batch_size, hidden_size]
        out = self.classifier[-1](hidden_states).squeeze(1)  # [batch_size]

        if labels is not None:
            loss = criterion(out, labels)
            return out, loss
        else:
            return out


try:  # For the case of reloading the model class
    for model in model_bag:
        model.__class__ = ElectraForNaiveTextDetection
except Exception:
    pass

In [ ]:
model_bag = []
for bag in range(train_dataset.bagging_size):
    try:
        model = ElectraForNaiveTextDetection()

        from safetensors.torch import load_file
        state_dict = load_file(f"./models/{PROJECT_NAME}_last/model.safetensors")
        model.load_state_dict(state_dict)
    except Exception:
        model = ElectraForNaiveTextDetection.from_pretrained(base_model_id)

    model_bag.append(model)
    model = model.bfloat16()
    model.to(device)

model_bag[0]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

def tokenize(batch):
    return tokenizer(
        batch, return_tensors="pt", return_token_type_ids=False,
        padding="longest" if len(batch) > 1 else False
    ).to(device)

In [ ]:
tokenize("Hello, my dog is cute")

## Train and Evaluate

### Utils

In [ ]:
def to_label(scores, threshold=0.5):
    return [1 if score > threshold else 0 for score in scores]

def calc_score(ls, lb, pd):
    def do_by_bag(ls, lb, pd):
        pd_discrete = to_label(pd)
        if len(ls) == 0:
            lss = 0.0
        else:
            lss = sum(ls) / len(ls)
        if len(lb) == 0:
            acc, f1, rocauc = 0.0, 0.0, 0.0
        else:
            acc = accuracy_score(lb, pd_discrete)
            f1 = f1_score(lb, pd_discrete)
            rocauc = roc_auc_score(lb, pd)
        return lss, acc, f1, rocauc

    lss, acc, f1, rocauc = zip(*[do_by_bag(ls[i], lb[i], pd[i]) for i in range(len(ls))])
    lss, acc, f1, rocauc = (" ".join("{:.4f}".format(i) for i in x) for x in (lss, acc, f1, rocauc))
    return f"Loss: {lss}, ACC: {acc}, F1: {f1}, ROCAUC: {rocauc}"

In [ ]:
BATCH_SIZE = 4, 8, 4

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE[0], shuffle=True, collate_fn=lambda batch: [tuple(zip(*x)) for x in tuple(zip(*batch))])
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE[1], shuffle=False, collate_fn=lambda x: tuple(zip(*x)))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE[2], shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

In [ ]:
EPOCHS = 5
START_EPOCH = 0
LEARNING_RATE = 5e-3
BACKBONE_LEARNING_RATE = 2e-5

optimizer = [optim.AdamW(model.classifier.parameters(), lr=LEARNING_RATE) for model in model_bag]
backbone_optimizer = [optim.AdamW(model.electra.parameters(), lr=BACKBONE_LEARNING_RATE) for model in model_bag]

In [ ]:
next(iter(train_loader))[0]

In [ ]:
next(iter(valid_loader))

In [ ]:
next(iter(test_loader))

### Training Loop

In [ ]:
with (
    tqdm(range(START_EPOCH, START_EPOCH+EPOCHS), desc="[Running Epochs]") as epochs,
    tqdm(range(len(train_dataset)//BATCH_SIZE[0]), desc="[Training]") as train_progress,
    tqdm(range(len(valid_dataset)//BATCH_SIZE[1]), desc="[Validating]") as valid_progress
):
    for epoch in epochs:
        train_progress.reset()
        train_loss, train_preds, train_labels, train_accums = ([[] for _ in range(train_dataset.bagging_size)] for _ in range(4))

        # Train
        for step, bagged_data in enumerate(train_loader):
            for bag, model, (texts, labels) in zip(range(1), model_bag, bagged_data):
                torch.cuda.empty_cache(); gc.collect();
                model.train()

                labels = torch.tensor(labels).to(device)
                input_ids = []
                attention_masks = []
                for tokenized, ori in zip(map(tokenize, texts), texts):
                    ids = tokenized['input_ids']
                    if len(tokenized['input_ids'][0]) > 512:
                        print(ids, file=sys.stderr)
                        #raise ValueError("Input length exceeds 512 tokens.", f"Input length: {len(ids[0])}", ori)
                    input_ids.append(tokenized['input_ids'].to(device))
                    attention_masks.append(tokenized['attention_mask'].to(device))

                try:
                    logits, loss = model(input_ids=input_ids, attention_mask=attention_masks, labels=labels.float())
                    scores = torch.sigmoid(logits).tolist()
                    train_preds[bag].extend(scores)
                    train_labels[bag].extend(labels.tolist())
                    train_loss[bag].append(loss.item())

                    loss.backward()
                    optimizer[bag].step()
                    backbone_optimizer[bag].step()
                    optimizer[bag].zero_grad()
                    backbone_optimizer[bag].zero_grad()

                    train_progress.update(1)
                    train_progress.set_description(f"[Training] " + calc_score(train_loss, train_labels, train_preds))
                except Exception as e:
                    if "CUDA" in str(e):
                        print(e, file=sys.stderr)
                    else:
                        pass#raise e

        # Validate
        valid_loss, valid_preds, valid_labels = ([[] for _ in range(train_dataset.bagging_size + 1)] for _ in range(3))
        valid_progress.reset()
        for texts, labels in valid_loader:
            torch.cuda.empty_cache(); gc.collect();
            labels = torch.tensor(labels).to(device)
            input_ids = []
            attention_masks = []
            for tokenized in [tokenize(t) for t in texts]:
                input_ids.append(tokenized['input_ids'].to(device))
                attention_masks.append(tokenized['attention_mask'].to(device))

            try:
                bagged_preds = [[] for _ in range(train_dataset.bagging_size)]
                for bag, model in zip(range(1), model_bag):
                    model.eval()
                    with torch.no_grad():
                        logits, loss = model(input_ids=input_ids, attention_mask=attention_masks, labels=labels.float())
                        scores = torch.sigmoid(logits).tolist()

                        valid_preds[bag].extend(scores)
                        bagged_preds[bag].extend(scores.item())
                        valid_labels[bag].extend(labels.tolist())
                        valid_loss[bag].append(loss.item())
                valid_preds[-1].extend(sum(p) / len(p) for p in zip(*bagged_preds))

                valid_progress.set_description(f"[Validating] " + calc_score(valid_loss, valid_labels*len(model_bag), valid_preds))
            except Exception as e:
                if "CUDA" in str(e):
                    print(e, file=sys.stderr)
                else:
                    pass#raise e

            valid_progress.update(1)

        model.save_pretrained(f"./models/{PROJECT_NAME}_{epoch}")
        #model.save_pretrained(f"./models/{PROJECT_NAME}_last")
        START_EPOCH = epoch

In [ ]:
# Model Saving
model.save_pretrained(f"./models/{PROJECT_NAME}_last")

### Final Output

In [ ]:
results = []
with tqdm(test_dataset_bundled, desc="[Testing]") as progress:
    humans, ais = 0, 0
    model.eval()
    for texts in progress:
        torch.cuda.empty_cache()
        gc.collect()

        try:
            with torch.no_grad():
                input_ids, attention_masks = [], []
                for tokenized in [tokenize(t) for t in texts]:
                    input_ids.append(tokenized['input_ids'].to(device))
                    attention_masks.append(tokenized['attention_mask'].to(device))

                logits = model(input_ids=input_ids, attention_mask=attention_masks)
                scores = torch.sigmoid(logits)
                results.extend(scores.tolist())
                for preds in to_label(scores):
                    if preds == 0:
                        humans += 1
                    else:
                        ais += 1
        except Exception as e:
            if "CUDA" in str(e):
                print(f"ERROR: {e} - {texts}")
            else:
                raise e

        progress.set_description(f"[Testing] Human: {humans/len(test_dataset):.2%}, Ai: {ais/len(test_dataset):.2%}")

len(results) == len(test_dataset)

In [ ]:
sub = pd.read_csv("./data/swuniv_dacon/" + test_dataset.submission_file, encoding='utf-8-sig')
sub

In [ ]:
sub['generated'] = results
sub

In [ ]:
plt.figure(figsize=(12, 7))
sns.histplot(data=sub, x="generated", kde=True, bins=50)
plt.title("Prediction Probability Distribution", fontsize=16)
plt.xlabel("Predicted Probability (Generated = 1)", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)

plt.show()
print(sub['generated'].describe())

In [ ]:
sub.to_csv(f"./data/submission_{PROJECT_NAME[2:]}.csv", index=False, encoding='utf-8-sig')